# Sales dataset
Load sales for 2025 and 2026. Keep this notebook and `sales_dataset.csv` in the same folder.

In [1]:
import pandas as pd

sales = pd.read_csv("sales_dataset.csv")
sales

,Year,Customer,product,Quantity,Price
0,2025,Customer A,Notebook,100,5.0
1,2025,Customer A,Pen,200,2.0
2,2025,Customer A,Folder,80,3.0
3,2025,Customer B,Notebook,150,5.0
4,2025,Customer B,Pen,300,2.0
5,2025,Customer B,Folder,100,3.0
6,2025,Customer C,Notebook,80,5.0
7,2025,Customer C,Pen,150,2.0
8,2025,Customer C,Folder,60,3.0
9,2025,Customer D,Notebook,120,5.0


## Classical price and volume/mix analysis

Following Sections 4.1–4.4 of `Price-Volume-Overview.ipynb`, 2025 is the previous year and 2026 is the current year. For this synthetic example, all customer–product relationships are assumed to be eligible base business, and fixed charges are zero.

- Sales = quantity × unit price.
- Average price = total sales / total quantity for each customer, product, and year (a quantity-weighted price if multiple transactions exist).
- Price effect = (average price 2026 − average price 2025) × quantity 2025.
- Volume mix effect = average price 2026 × (quantity 2026 − quantity 2025).

The price–quantity interaction is included in volume/mix, as in the overview. Each row must reconcile: **Sales 2026 − Sales 2025 = Price effect + Volume mix effect**.

In [2]:
import pandas as pd

sales = pd.read_csv("sales_dataset.csv")

# Aggregate transactions before calculating weighted average prices.
annual_sales = (
    sales.rename(columns={"product": "Product"})
    .assign(Sales=sales["Quantity"] * sales["Price"])
    .groupby(["Customer", "Product", "Year"], as_index=False)
    .agg(Sales=("Sales", "sum"), Qty=("Quantity", "sum"))
)

# This example requires comparable business and positive quantities in both years.
previous = annual_sales.loc[annual_sales["Year"].eq(2025)].drop(columns="Year")
current = annual_sales.loc[annual_sales["Year"].eq(2026)].drop(columns="Year")
comparison = previous.merge(
    current, on=["Customer", "Product"], how="outer",
    suffixes=(" 2025", " 2026"), validate="one_to_one", indicator=True,
)
if not comparison["_merge"].eq("both").all():
    raise ValueError("New or lost relationships require separate volume/mix treatment.")
if not comparison[["Qty 2025", "Qty 2026"]].gt(0).all().all():
    raise ValueError("Positive quantities are required to calculate average prices.")

comparison["Avrg. price 2025"] = comparison["Sales 2025"] / comparison["Qty 2025"]
comparison["Avrg. price 2026"] = comparison["Sales 2026"] / comparison["Qty 2026"]
comparison["Price effect"] = (
    (comparison["Avrg. price 2026"] - comparison["Avrg. price 2025"])
    * comparison["Qty 2025"]
)
comparison["Volume mix effect"] = (
    comparison["Avrg. price 2026"]
    * (comparison["Qty 2026"] - comparison["Qty 2025"])
)

columns = [
    "Customer", "Product", "Sales 2025", "Sales 2026", "Qty 2025", "Qty 2026",
    "Avrg. price 2025", "Avrg. price 2026", "Price effect", "Volume mix effect",
]
price_volume_table = (
    comparison[columns]
    .sort_values(["Customer", "Product"])
    .reset_index(drop=True)
)

# Check every relationship before rounding the displayed values.
reconciliation_error = (
    price_volume_table["Sales 2026"] - price_volume_table["Sales 2025"]
    - price_volume_table["Price effect"] - price_volume_table["Volume mix effect"]
)
assert reconciliation_error.abs().lt(1e-8).all(), "Sales bridge does not reconcile."

money_columns = [
    "Sales 2025", "Sales 2026", "Avrg. price 2025", "Avrg. price 2026",
    "Price effect", "Volume mix effect",
]
price_volume_table.style.format(
    {**{column: "{:,.2f}" for column in money_columns},
     "Qty 2025": "{:,.0f}", "Qty 2026": "{:,.0f}"}
).hide(axis="index")

Customer,Product,Sales 2025,Sales 2026,Qty 2025,Qty 2026,Avrg. price 2025,Avrg. price 2026,Price effect,Volume mix effect
Customer A,Folder,240.00,252.00,80,90,3.00,2.80,-16.00,28.00
Customer A,Notebook,500.00,660.00,100,120,5.00,5.50,50.00,110.00
Customer A,Pen,400.00,396.00,200,180,2.00,2.20,40.00,-44.00
Customer B,Folder,300.00,280.00,100,100,3.00,2.80,-20.00,0.00
Customer B,Notebook,750.00,770.00,150,140,5.00,5.50,75.00,-55.00
Customer B,Pen,600.00,726.00,300,330,2.00,2.20,60.00,66.00
Customer C,Folder,180.00,140.00,60,50,3.00,2.80,-12.00,-28.00
Customer C,Notebook,400.00,550.00,80,100,5.00,5.50,40.00,110.00
Customer C,Pen,300.00,330.00,150,150,2.00,2.20,30.00,0.00
Customer D,Folder,270.00,294.00,90,105,3.00,2.80,-18.00,42.00


In [3]:

total_sales_2025 = price_volume_table["Sales 2025"].sum()
total_sales_2026 = price_volume_table["Sales 2026"].sum()
total_price_effect = price_volume_table["Price effect"].sum()
total_volume_mix_effect = price_volume_table["Volume mix effect"].sum()
sales_change = total_sales_2026 - total_sales_2025
assert abs(sales_change - total_price_effect - total_volume_mix_effect) < 1e-8

print(f"Sales 2025:        {total_sales_2025:,.2f}")
print(f"Sales 2026:        {total_sales_2026:,.2f}")
print(f"Sales change:     {sales_change:,.2f}")
print(f"Price effect:     {total_price_effect:,.2f}")
print(f"Volume mix effect:{total_volume_mix_effect:,.2f}")
print("Reconciliation passed for every customer–product pair and overall.")

Sales 2025:        5,040.00
Sales 2026:        5,619.00
Sales change:     579.00
Price effect:     339.00
Volume mix effect:240.00
Reconciliation passed for every customer–product pair and overall.


## Derivatives: revenue sensitivities (Section 6.1)

For $S(P,V,F)=PV+F$, differentiate one input while holding the others constant:

$$\frac{\partial S}{\partial P}=V,\qquad
\frac{\partial S}{\partial V}=P,\qquad
\frac{\partial S}{\partial F}=1.$$

The code uses **SymPy** to derive these expressions, then evaluates them for each customer–product pair in 2025 and 2026. The price sensitivity has units of quantity; the volume sensitivity is revenue per additional unit; fixed-charge sensitivity is revenue per currency unit of charge. These are sensitivities, rather than revenue effects or predictions of demand.

For finite changes, applying the starting sensitivities gives $V_1\Delta P+P_1\Delta V+\Delta F$. The exact change also contains $\Delta P\Delta V$. The second table compares these amounts and applies the classical path: price sensitivity at previous quantity, then volume sensitivity at current price. Fixed charges are zero in this dataset, and all pairs are assumed to be eligible base business.

This cell loads the CSV independently and needs pandas, NumPy, SymPy, and IPython. Run it with the notebook folder as the working directory.

In [4]:
import pandas as pd
import numpy as np
import sympy as sp
from IPython.display import display, Math

P, V, F = sp.symbols("P V F", real=True)
revenue = P * V + F
dS_dP = sp.diff(revenue, P)
dS_dV = sp.diff(revenue, V)
dS_dF = sp.diff(revenue, F)
display(Math(r"S(P,V,F)=" + sp.latex(revenue)))
for label, derivative in [
    (r"\frac{\partial S}{\partial P}", dS_dP),
    (r"\frac{\partial S}{\partial V}", dS_dV),
    (r"\frac{\partial S}{\partial F}", dS_dF),
]:
    display(Math(label + "=" + sp.latex(derivative)))

derivative_sales = pd.read_csv("sales_dataset.csv").rename(columns={"product": "Product"})
derivative_sales["Sales"] = derivative_sales["Quantity"] * derivative_sales["Price"]
yearly = derivative_sales.groupby(["Customer", "Product", "Year"]).agg(
    Sales=("Sales", "sum"), Qty=("Quantity", "sum")
)
base = yearly.xs(2025, level="Year")
current = yearly.xs(2026, level="Year")
if not base.index.equals(current.index):
    raise ValueError("Derivative comparison requires matching customer-product pairs.")
if not base["Qty"].gt(0).all() or not current["Qty"].gt(0).all():
    raise ValueError("Positive quantities are required for weighted average prices.")


P1, P2 = base["Sales"] / base["Qty"], current["Sales"] / current["Qty"]
V1, V2 = base["Qty"], current["Qty"]
F1 = F2 = 0.0  # No fixed charges in the synthetic dataset.
delta_P, delta_V, delta_F = P2 - P1, V2 - V1, F2 - F1

price_sensitivity = sp.lambdify((P, V, F), dS_dP, "numpy")
volume_sensitivity = sp.lambdify((P, V, F), dS_dV, "numpy")
charge_sensitivity = sp.lambdify((P, V, F), dS_dF, "numpy")
sensitivities = pd.DataFrame(index=base.index)
sensitivities["dS/dP 2025"] = price_sensitivity(P1, V1, F1)
sensitivities["dS/dP 2026"] = price_sensitivity(P2, V2, F2)
sensitivities["dS/dV 2025"] = volume_sensitivity(P1, V1, F1)
sensitivities["dS/dV 2026"] = volume_sensitivity(P2, V2, F2)
sensitivities["dS/dF (both years)"] = charge_sensitivity(P1, V1, F1)
print("Revenue sensitivities by customer and product")
display(sensitivities.style.format("{:,.2f}"))


derivative_effects = pd.DataFrame(index=base.index)
derivative_effects["Delta P"] = delta_P
derivative_effects["Delta V"] = delta_V
derivative_effects["First-order change (2025)"] = (
    sensitivities["dS/dP 2025"] * delta_P
    + sensitivities["dS/dV 2025"] * delta_V
    + sensitivities["dS/dF (both years)"] * delta_F
)
derivative_effects["Interaction"] = delta_P * delta_V
derivative_effects["Actual sales change"] = current["Sales"] - base["Sales"]
derivative_effects["Price effect"] = (
    sensitivities["dS/dP 2025"] * delta_P
    + sensitivities["dS/dF (both years)"] * delta_F
)
derivative_effects["Volume mix effect"] = sensitivities["dS/dV 2026"] * delta_V

# Validate both the finite-change expansion and the classical comparison path.
np.testing.assert_allclose(
    derivative_effects["First-order change (2025)"] + derivative_effects["Interaction"],
    derivative_effects["Actual sales change"], atol=1e-8, rtol=0,
)
np.testing.assert_allclose(
    derivative_effects["Price effect"] + derivative_effects["Volume mix effect"],
    derivative_effects["Actual sales change"], atol=1e-8, rtol=0,
)
print("Finite changes and the classical path (amounts in the dataset currency)")
display(derivative_effects.style.format("{:,.2f}"))
print(f"Starting-sensitivity approximation: {derivative_effects['First-order change (2025)'].sum():,.2f}")
print(f"Price-quantity interaction: {derivative_effects['Interaction'].sum():,.2f}")
print(f"Actual sales change: {derivative_effects['Actual sales change'].sum():,.2f}")
print(f"Classical price effect: {derivative_effects['Price effect'].sum():,.2f}")
print(f"Classical volume/mix effect: {derivative_effects['Volume mix effect'].sum():,.2f}")


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Revenue sensitivities by customer and product


Finite changes and the classical path (amounts in the dataset currency)


Starting-sensitivity approximation: 564.00
Price-quantity interaction: 15.00
Actual sales change: 579.00
Classical price effect: 339.00
Classical volume/mix effect: 240.00


## Integrals: exact revenue effects (Section 6.2)

Integrate the sensitivities along the same comparison path as the classical calculation:

$$\text{Price effect}=\int_{P_1}^{P_2}V_1\,dP+\int_{F_1}^{F_2}1\,dF,$$
$$\text{Volume mix effect}=\int_{V_1}^{V_2}P_2\,dV.$$

Hold previous quantity constant during repricing, then hold current price constant while changing quantity. Integrate the **sensitivities**, rather than revenue, over each input, rather than time. These constant integrands yield exact amounts, including negative effects when the limits decrease. The interaction is already included in volume/mix.

The cell below derives the definite integrals with SymPy and evaluates them for each customer–product pair. As before, all pairs are assumed to be eligible base business and dataset fixed charges are zero. It also checks the overview's worked example, where fixed charges change. The cell loads the CSV independently, so it can run in a fresh kernel.

In [23]:
import pandas as pd
import numpy as np
import sympy as sp
from IPython.display import display, Math

P, V, F = sp.symbols("P V F", real=True)
P_old, P_new, V_old, V_new, F_old, F_new = sp.symbols(
    "P_1 P_2 V_1 V_2 F_1 F_2", real=True
)

S = P * V + F
variable_price_integral = sp.integrate(
    sp.diff(S, P).subs(V, V_old), (P, P_old, P_new)
)

fixed_charge_integral = sp.integrate(sp.diff(S, F), (F, F_old, F_new))
volume_integral = sp.integrate(
    sp.diff(S, V).subs(P, P_new), (V, V_old, V_new)
)
for label, result in [
    (r"\int_{P_1}^{P_2}V_1\,dP", variable_price_integral),
    (r"\int_{F_1}^{F_2}1\,dF", fixed_charge_integral),
    (r"\int_{V_1}^{V_2}P_2\,dV", volume_integral),
]:
    display(Math(label + "=" + sp.latex(sp.factor(result))))
integral_inputs = (P_old, P_new, V_old, V_new, F_old, F_new)
evaluate_price_integral = sp.lambdify(integral_inputs, variable_price_integral, "numpy")
evaluate_charge_integral = sp.lambdify(integral_inputs, fixed_charge_integral, "numpy")
evaluate_volume_integral = sp.lambdify(integral_inputs, volume_integral, "numpy")

integral_sales = pd.read_csv("sales_dataset.csv").rename(columns={"product": "Product"})
integral_sales["Sales"] = integral_sales["Quantity"] * integral_sales["Price"]
integral_annual = integral_sales.groupby(["Customer", "Product", "Year"]).agg(
    Sales=("Sales", "sum"), Qty=("Quantity", "sum")
)   

integral_previous = integral_annual.xs(2025, level="Year")
integral_current = integral_annual.xs(2026, level="Year")
if not integral_previous.index.equals(integral_current.index):
    raise ValueError("Integral comparison requires matching customer-product pairs.")
if not integral_previous["Qty"].gt(0).all() or not integral_current["Qty"].gt(0).all():
    raise ValueError("Positive quantities are required for weighted average prices.")

old_price = integral_previous["Sales"] / integral_previous["Qty"]
new_price = integral_current["Sales"] / integral_current["Qty"]
old_qty, new_qty = integral_previous["Qty"], integral_current["Qty"]
values = (old_price, new_price, old_qty, new_qty, 0.0, 0.0)

integral_effects = pd.DataFrame(index=integral_previous.index)
integral_effects["Variable price integral"] = evaluate_price_integral(*values)
integral_effects["Fixed charge integral"] = evaluate_charge_integral(*values)
integral_effects["Price effect"] = (
    integral_effects["Variable price integral"] + integral_effects["Fixed charge integral"]
)

integral_effects["Volume mix effect"] = evaluate_volume_integral(*values)
integral_effects["Total integral change"] = (
    integral_effects["Price effect"] + integral_effects["Volume mix effect"]
)
integral_effects["Actual sales change"] = integral_current["Sales"] - integral_previous["Sales"]


np.testing.assert_allclose(
    integral_effects["Price effect"], (new_price - old_price) * old_qty,
    atol=1e-8, rtol=0,
)
np.testing.assert_allclose(
    integral_effects["Volume mix effect"], new_price * (new_qty - old_qty),
    atol=1e-8, rtol=0,
)
np.testing.assert_allclose(
    integral_effects["Total integral change"], integral_effects["Actual sales change"],
    atol=1e-8, rtol=0,
)
print("Exact integral effects by customer and product")
display(integral_effects.style.format("{:,.2f}"))
print("Dataset totals")
print(f"Price effect: {integral_effects['Price effect'].sum():,.2f}")
print(f"Volume/mix effect: {integral_effects['Volume mix effect'].sum():,.2f}")
print(f"Exact sales change: {integral_effects['Total integral change'].sum():,.2f}")

example_values = (10, 11, 100, 120, 200, 220)
example_variable_price = evaluate_price_integral(*example_values)
example_fixed_charge = evaluate_charge_integral(*example_values)
example_volume = evaluate_volume_integral(*example_values)
example_total = example_variable_price + example_fixed_charge + example_volume
np.testing.assert_allclose(
    [example_variable_price, example_fixed_charge, example_volume, example_total],
    [100, 20, 220, (11 * 120 + 220) - (10 * 100 + 200)],
    atol=1e-8, rtol=0,
)
print("\nOverview example (EUR)")
print(f"Variable price integral: {example_variable_price:,.2f}")
print(f"Fixed charge integral: {example_fixed_charge:,.2f}")
print(f"Price effect: {example_variable_price + example_fixed_charge:,.2f}")
print(f"Volume/mix effect: {example_volume:,.2f}")
print(f"Exact sales change: {example_total:,.2f}")


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Exact integral effects by customer and product


Dataset totals
Price effect: 339.00
Volume/mix effect: 240.00
Exact sales change: 579.00

Overview example (EUR)
Variable price integral: 100.00
Fixed charge integral: 20.00
Price effect: 120.00
Volume/mix effect: 220.00
Exact sales change: 340.00
